# Description

## This notebook is used to extract the extract emails of potential profiles and create a outreach list from top VC companies.

In [13]:
%load_ext autoreload
%autoreload 2

import logging
import os

import numpy as np

import ck_marketing.hunterio.hunterapi as cmahuhun
import ck_marketing.linkedin.profile_filtering as cmliprfi
import helpers.hgoogle_file_api as hgfiapi
from ck_marketing.hunterio.hunterapi import GoogleSheetsHelper, HunterIO
from ck_marketing.linkedin.phantombuster_api import Phantom
import helpers.hdbg as hdbg
import helpers.henv as henv
import helpers.hprint as hprint

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [14]:
# Configure logger.
hdbg.init_logger(verbosity=logging.INFO)
_LOG = logging.getLogger(__name__)

# Print system signature.
_LOG.info("%s", henv.get_system_signature()[0])

# Configure the notebook style.
hprint.config_notebook()

DEBUG:helpers.hsystem:> (git branch --show-current) 2>&1
DEBUG:helpers.hsystem:> (git rev-parse --short HEAD) 2>&1
DEBUG:helpers.hsystem:> (git log --date=local --oneline --graph --date-order --decorate --pretty=format:'%h %<(8)%aN%  %<(65)%s (%>(14)%ar) %ad %<(10)%d' -3) 2>&1
INFO:__main__:# Git
  branch_name='Cmamptask8909_Collect_emails_Tier1_VCs'
  hash='48164d481'
  # Last commits:
    *   48164d481 shaunak01 Merge branch 'master' into Cmamptask8909_Collect_emails_Tier1_VCs (24 minutes ago) Mon Jul 15 13:45:14 2024  (HEAD -> Cmamptask8909_Collect_emails_Tier1_VCs)
    |\  
    | * 936bd502e Juraj Smeriga CmampTask8680_Group_HW_resource_usage_of_ECS_tasks_by_type_of_workload (#9033) (   5 hours ago) Mon Jul 15 09:17:08 2024  (origin/master, origin/HEAD, master)
    | * a3000a27b pavolrabatin CmTask8737_Modify_RDS_Terraform_module_with_dynamic_per_instance_DB_username_and_password_variables (#8936) (    2 days ago) Sat Jul 13 18:59:32 2024           
# Machine info
  system=Linux
  

## Extracting Profiles - Phantom Bustor

In [12]:
# Get the API keys from the environment variables.
phantom_api_key = os.getenv("Phantom_API_KEY")
hunter_api_key = os.getenv("Hunter_API_KEY")

In [13]:
# Initialize the Phantom instance.
phantom = Phantom(phantom_api_key)

In [14]:
# Get and print all agents and their IDs.
agents = phantom.get_all_agents()
print("List of all agents and their IDs:\n")
for agent in agents:
    print(f"Agent Name: {agent['name']}, Agent ID: {agent['id']}")

List of all agents and their IDs:

Agent Name: Bessemer Venture Partners, Agent ID: 7890616458359264


In [15]:
# Get agent ID and name.
AGENT_ID = "7890616458359264"

In [16]:
specific_agent_name = phantom.get_agent_name(AGENT_ID)
print(f"Selected Phantom: {specific_agent_name}")

Selected Phantom: Bessemer Venture Partners


In [17]:
# Google Drive Setup.
google_creds_path = "service.json"
google_sheet_helper = GoogleSheetsHelper(google_creds_path)
drive_folder_id = "1utxjyRBuLR1RxCpX_B5xm6fC5bYYp7Ex"
sheet_name = f"{specific_agent_name}_search_export"
tab_name = "search_export"

In [18]:
# Launch the agent and get the results in a DataFrame
phantom.launch_agent(AGENT_ID)
result_response_json = phantom.fetch_agent_results(AGENT_ID)

hiishaun
hiishaun


In [19]:
csv_url = phantom.get_csv_url(result_response_json.get("output", ""))
df = phantom.download_csv(csv_url)
df = df.replace([np.nan, np.inf, -np.inf], "", inplace=False)
print("DataFrame is fetched")
# print(df.head(2))

DataFrame is fetched


In [20]:
# Create the Google Sheet and get the file ID.
file_id = hgfiapi.create_empty_google_file("sheet", sheet_name, drive_folder_id)

# Initialize Google Sheets Helper.
google_sheets_helper = GoogleSheetsHelper(google_creds_path)
sheet = google_sheets_helper.google_account.open_by_key(file_id)
default_worksheet = sheet.get_worksheet(0)
new_tab_name = "search_export"
default_worksheet.update_title(new_tab_name)
# Write the DataFrame to the renamed default tab.
google_sheets_helper.write_results(file_id, df, new_tab_name)
print(
    f"DataFrame written to Google Sheet '{new_tab_name}' in file ID '{file_id}' successfully."
)

INFO:helpers.hgoogle_file_api:Created a new Google sheet 'Bessemer Venture Partners_search_export'.
/venv/lib/python3.9/site-packages/gspread/worksheet.py:1069: UserWarning: [Deprecated][in version 6.0.0]: method signature will change to: 'Worksheet.update(value = [[]], range_name=)' arguments 'range_name' and 'values' will swap, values will be mandatory of type: 'list(list(...))'
  warnings.warn(
INFO:ck_marketing.hunterio.hunterapi:Email extraction completed. Results saved in the new tab: search_export


DataFrame written to Google Sheet 'search_export' in file ID '1xNBBDcQ6Y2EsmE6DQ9D9UGh50nm1qWu3Bqoo3oFFtYM' successfully.


## Clean Profiles - Filtering

In [21]:
words = []

In [22]:
filtered_df = cmliprfi.filter_df(df, "title", words, "keep")

INFO:ck_marketing.linkedin.profile_filtering:Filtered dataframe to keep rows where 'title' contains any of [].
INFO:ck_marketing.linkedin.profile_filtering:24 entries were kept.
INFO:ck_marketing.linkedin.profile_filtering:Entries before filter: 24, Entries after filter: 24
INFO:ck_marketing.linkedin.profile_filtering:Original entries: 24
INFO:ck_marketing.linkedin.profile_filtering:Remaining entries after filtering: 24
INFO:ck_marketing.linkedin.profile_filtering:Removed entries: 0
INFO:ck_marketing.linkedin.profile_filtering:Percentage of entries removed: 0.00%


In [23]:
cleaned_profiles_tab = sheet.add_worksheet(
    title="cleaned_profiles", rows="100", cols="20"
)
# Write the filtered DataFrame to the new tab.
google_sheet_helper.write_results(file_id, filtered_df, "cleaned_profiles")
print(
    f"Filtered DataFrame written to new tab 'cleaned_profiles' in Google Sheet with file ID '{file_id}' successfully."
)

INFO:ck_marketing.hunterio.hunterapi:Email extraction completed. Results saved in the new tab: cleaned_profiles


Filtered DataFrame written to new tab 'cleaned_profiles' in Google Sheet with file ID '1xNBBDcQ6Y2EsmE6DQ9D9UGh50nm1qWu3Bqoo3oFFtYM' successfully.


## Extracting emails - HunterIO

In [24]:
if False:
    first_name_col = "firstName"
    last_name_col = "lastName"
    company_col = "companyName"
    tab_name = "cleaned_profiles"

    cmahuhun.process_records(
        api_key=hunter_api_key,
        google_creds_path=google_creds_path,
        file_id=file_id,
        first_name_col=first_name_col,
        last_name_col=last_name_col,
        company_col=company_col,
        tab_name=tab_name,
    )

INFO:ck_marketing.hunterio.hunterapi:Starting process to read records and find emails
INFO:ck_marketing.hunterio.hunterapi:Finding bulk emails using company name
INFO:ck_marketing.hunterio.hunterapi:Writing results to Google Sheets
/venv/lib/python3.9/site-packages/gspread/worksheet.py:1069: UserWarning: [Deprecated][in version 6.0.0]: method signature will change to: 'Worksheet.update(value = [[]], range_name=)' arguments 'range_name' and 'values' will swap, values will be mandatory of type: 'list(list(...))'
  warnings.warn(
INFO:ck_marketing.hunterio.hunterapi:Email extraction completed. Results saved in the new tab: hunter_results
INFO:ck_marketing.hunterio.hunterapi:Total records processed: 24
INFO:ck_marketing.hunterio.hunterapi:Emails found: 23
INFO:ck_marketing.hunterio.hunterapi:Emails not found: 1
INFO:ck_marketing.hunterio.hunterapi:Number of unique companies: 1
INFO:ck_marketing.hunterio.hunterapi:Percentage of emails found: 95.83%
INFO:ck_marketing.hunterio.hunterapi:Compa

## Email Verification - HunterIO

In [ ]:
if False:
    df_drop = google_sheet_helper.read_sheet(file_id, "hunter_results")
    hunter_instance = HunterIO(hunter_api_key)
    verified_df = hunter_instance.verify_emails(df_drop, "hunter_extracted_email")
    cleaned_profiles_tab = sheet.add_worksheet(
        title="hunter_verification", rows="100", cols="20"
    )
    google_sheet_helper.write_results(file_id, verified_df, "hunter_verification")
    print(
        f"Filtered DataFrame written to new tab in Google Sheet with file ID '{file_id}' successfully."
    )

## Final dataframe

In [27]:
final_df = verified_df[
    [
        "fullName",
        "profileUrl",
        "title",
        "hunter_extracted_email",
        "hunter_verification",
    ]
]
# Step 2: Filter out rows where 'hunter_extracted_email' is empty.
final_df = final_df[
    final_df["hunter_extracted_email"].notna()
    & (final_df["hunter_extracted_email"] != "")
]

cleaned_profiles_tab = sheet.add_worksheet(
    title="final_df", rows="100", cols="20"
)
google_sheet_helper.write_results(file_id, final_df, "final_df")
print(
    f"Filtered DataFrame written to new tab in Google Sheet with file ID '{file_id}' successfully."
)

/venv/lib/python3.9/site-packages/gspread/worksheet.py:1069: UserWarning: [Deprecated][in version 6.0.0]: method signature will change to: 'Worksheet.update(value = [[]], range_name=)' arguments 'range_name' and 'values' will swap, values will be mandatory of type: 'list(list(...))'
  warnings.warn(
INFO:ck_marketing.hunterio.hunterapi:Email extraction completed. Results saved in the new tab: final_df


Filtered DataFrame written to new tab in Google Sheet with file ID '1xNBBDcQ6Y2EsmE6DQ9D9UGh50nm1qWu3Bqoo3oFFtYM' successfully.
